# 03 — Day Validity Classification

This notebook mirrors `python/03_valid_day_classification.py` with richer documentation and inline visualisations.

## What this step does

Each (participant × session × day) is classified as **valid** or **invalid** based on wear-time completeness. Days that fail either criterion are excluded from feature extraction but are **not dropped** from the dataset.

### Day validity criteria

| Criterion | Threshold | Rationale |
|-----------|-----------|----------|
| `n_hr_slots` ≥ 8 | of 12 possible 2-hr slots | Ensures ≥ 67 % wear coverage |
| `has_nighttime` = True | ≥ 1 non-NaN HR slot in 10 pm–6 am window | Required for sleep feature computation |

### Wave-level exclusions

After per-day classification, participants are excluded from specific analyses if:
- `n_valid_days < 14` → excluded from clustering entirely
- `n_valid_weekdays < 8` → excluded from weekday-specific features
- `n_valid_weekend_days < 4` → excluded from weekend-specific features

**Input:** `data/processed/imputed/imputed_data.parquet`  
**Output:** `data/processed/day_validity.parquet`, `data/handoff/wear_quality.csv`

In [ ]:
import sys
from pathlib import Path

# Allow imports from python/ when running notebook from python/notebooks/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import (
    IMPUTED_PARQUET, DAY_VALIDITY_PARQUET, HANDOFF_DIR, LOGS_DIR,
    SESSION_LABELS,
    COL_ID, COL_SESSION, COL_DAY, COL_WKND, COL_HR,
    NIGHTTIME_HOURS, MIN_HR_SLOTS_PER_DAY,
    MIN_VALID_DAYS, MIN_VALID_WEEKDAYS, MIN_VALID_WEEKEND_DAYS,
)

LOGS_DIR.mkdir(parents=True, exist_ok=True)
DAY_VALIDITY_PARQUET.parent.mkdir(parents=True, exist_ok=True)
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "03_valid_day_classification.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

print(f"Validity thresholds:")
print(f"  Min HR slots/day:       {MIN_HR_SLOTS_PER_DAY}")
print(f"  Nighttime hours (set):  {sorted(NIGHTTIME_HOURS)}")
print(f"  Min valid days:         {MIN_VALID_DAYS}")
print(f"  Min valid weekdays:     {MIN_VALID_WEEKDAYS}")
print(f"  Min valid weekend days: {MIN_VALID_WEEKEND_DAYS}")

## Load imputed data

In [ ]:
log.info(f"Loading {IMPUTED_PARQUET}")
df = pd.read_parquet(IMPUTED_PARQUET)
log.info(f"  Input rows: {len(df):,}")
assert COL_HR in df.columns

print(f"Shape: {df.shape}")
print(f"Participants: {df[COL_ID].nunique():,}")
print(f"Sessions: {df[COL_SESSION].unique()}")
print(f"\nFirst few rows:")
df.head(3)

## Classify days

For each (participant, session, day) group we compute:
- `n_hr_slots` — count of non-NaN HR observations
- `has_nighttime` — whether any nighttime slot (10 pm–6 am) has a valid HR reading

A day is marked `is_valid = True` only when both criteria are met.

In [ ]:
day_stats = (
    df.groupby([COL_ID, COL_SESSION, COL_DAY, COL_WKND])
    .apply(
        lambda g: pd.Series({
            "n_hr_slots":    int(g[COL_HR].notna().sum()),
            "has_nighttime": bool(
                g.loc[g["start_hour"].isin(NIGHTTIME_HOURS), COL_HR].notna().any()
            ),
        })
    )
    .reset_index()
)

day_stats["is_valid"] = (
    (day_stats["n_hr_slots"] >= MIN_HR_SLOTS_PER_DAY) &
    day_stats["has_nighttime"]
)

n_valid = day_stats["is_valid"].sum()
n_total = len(day_stats)
log.info(f"  Valid days: {n_valid:,} / {n_total:,} ({n_valid / n_total:.1%})")

print(f"Total days:   {n_total:,}")
print(f"Valid days:   {n_valid:,} ({n_valid / n_total:.1%})")
print(f"Invalid days: {n_total - n_valid:,}")

# Break down by session
print(f"\nValidity by session:")
for sess, label in SESSION_LABELS.items():
    sub = day_stats[day_stats[COL_SESSION] == sess]
    if len(sub) == 0:
        continue
    nv = sub["is_valid"].sum()
    nt = len(sub)
    print(f"  {label} ({sess}): {nv:,} / {nt:,} ({nv / nt:.1%})")

In [ ]:
# Distribution of number of valid days per participant per session
valid_day_counts = (
    day_stats[day_stats["is_valid"]]
    .groupby([COL_ID, COL_SESSION])[COL_DAY]
    .count()
    .reset_index(name="n_valid_days")
)

sessions_present = valid_day_counts[COL_SESSION].unique()
n_sess = len(sessions_present)

fig, axes = plt.subplots(1, n_sess, figsize=(6 * n_sess, 4), sharey=False)
if n_sess == 1:
    axes = [axes]

for ax, sess in zip(axes, sessions_present):
    label = SESSION_LABELS.get(sess, sess)
    sub = valid_day_counts[valid_day_counts[COL_SESSION] == sess]["n_valid_days"]
    ax.hist(sub, bins=range(0, sub.max() + 2), color="steelblue", edgecolor="white", linewidth=0.4)
    ax.axvline(MIN_VALID_DAYS, color="red", linestyle="--", label=f"threshold = {MIN_VALID_DAYS}")
    ax.set_title(f"Valid days per participant — {label}")
    ax.set_xlabel("n_valid_days")
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.show()

## Wave-level quality summary

We aggregate day-level validity to a per-participant per-session quality summary and apply exclusion flags. `valid_day_dispersion` (SD of valid-day index positions) captures whether valid days are spread evenly across the recording window.

In [ ]:
def _summarise_wave(grp: pd.DataFrame) -> pd.Series:
    valid   = grp[grp["is_valid"]]
    n_valid = len(valid)
    n_wkday = int((~valid[COL_WKND]).sum())
    n_wkend = int(valid[COL_WKND].sum())

    valid_indices = np.where(grp["is_valid"].values)[0]
    dispersion = float(np.std(valid_indices)) if len(valid_indices) > 1 else 0.0

    n_days = len(grp)
    return pd.Series({
        "n_valid_days":         n_valid,
        "n_valid_weekdays":     n_wkday,
        "n_valid_weekend_days": n_wkend,
        "wear_time_fraction":   n_valid / n_days if n_days > 0 else 0.0,
        "valid_day_dispersion": dispersion,
    })

wave_quality = (
    day_stats.groupby([COL_ID, COL_SESSION])
    .apply(_summarise_wave)
    .reset_index()
)

# Apply exclusion flags
wave_quality["exclude_clustering"]       = wave_quality["n_valid_days"] < MIN_VALID_DAYS
wave_quality["exclude_weekday_features"] = wave_quality["n_valid_weekdays"] < MIN_VALID_WEEKDAYS
wave_quality["exclude_weekend_features"] = wave_quality["n_valid_weekend_days"] < MIN_VALID_WEEKEND_DAYS
wave_quality["wave_label"]               = wave_quality[COL_SESSION].map(SESSION_LABELS)

log.info(f"  Excluded from clustering:       {wave_quality['exclude_clustering'].sum()}")
log.info(f"  Excluded from weekday features: {wave_quality['exclude_weekday_features'].sum()}")
log.info(f"  Excluded from weekend features: {wave_quality['exclude_weekend_features'].sum()}")

print(f"Participants in wave_quality: {len(wave_quality):,}")
print(f"  Excluded from clustering:       {wave_quality['exclude_clustering'].sum():,}")
print(f"  Excluded from weekday features: {wave_quality['exclude_weekday_features'].sum():,}")
print(f"  Excluded from weekend features: {wave_quality['exclude_weekend_features'].sum():,}")

In [ ]:
# Summary table: mean +/- SD of n_valid_days and wear_time_fraction per session
summary_rows = []
for sess in wave_quality[COL_SESSION].unique():
    label = SESSION_LABELS.get(sess, sess)
    sub = wave_quality[wave_quality[COL_SESSION] == sess]
    summary_rows.append({
        "session":            label,
        "n_participants":     len(sub),
        "n_valid_days_mean":  f"{sub['n_valid_days'].mean():.1f}",
        "n_valid_days_sd":    f"{sub['n_valid_days'].std():.1f}",
        "wear_frac_mean":     f"{sub['wear_time_fraction'].mean():.3f}",
        "wear_frac_sd":       f"{sub['wear_time_fraction'].std():.3f}",
        "excl_clustering":    int(sub['exclude_clustering'].sum()),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## Save

In [ ]:
day_stats.to_parquet(DAY_VALIDITY_PARQUET, index=False)
log.info(f"Saved day validity → {DAY_VALIDITY_PARQUET}")

wave_quality.to_csv(HANDOFF_DIR / "wear_quality.csv", index=False)
log.info(f"Saved wear quality → {HANDOFF_DIR / 'wear_quality.csv'}")

print(f"Saved day_validity.parquet → {DAY_VALIDITY_PARQUET}  ({len(day_stats):,} rows)")
print(f"Saved wear_quality.csv     → {HANDOFF_DIR / 'wear_quality.csv'}  ({len(wave_quality):,} rows)")